# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Get the dataset metadata as a Python dict
metadata = json.loads(dataset.metadata.to_json())
print(f"{metadata.get('name', '<no name>')}\n{metadata.get('description', '<no description>')}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# Find all available record sets
record_sets = []
if 'recordSet' in metadata and metadata['recordSet']:
    # recordSet can be a list or dict
    record_sets = metadata['recordSet'] if isinstance(metadata['recordSet'], list) else [metadata['recordSet']]
    record_set_ids = []
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs
        print(f"  Record set @id: {rs_id}")
        record_set_ids.append(rs_id)
else:
    # If recordSet is not at top level, check distribution
    dist = metadata.get('distribution', [])
    if dist:
        from urllib.request import urlopen
        # Try to open the Croissant distribution and parse recordSets
        # Not all Croissant files have this, so this is best effort only.
        croissant_url = dist[0]['@id'] if isinstance(dist[0], dict) else dist[0]
        print(f"Fetching main recordSet schema from: {croissant_url}")
        # Print guidance, manual determination required
        print("\n(Note: In this dataset, record sets may need to be inspected from the Croissant file's schema details.)")
        # You can open the Croissant JSON-LD file and inspect `recordSet` entries
        # For this notebook, we will use the mlcroissant API introspection:
        pass

# Use mlcroissant to list available record sets and their fields, always referencing `@id`
print("\nDiscovered record sets via mlcroissant:")
for recset in dataset.record_sets:
    print(f"- Record set @id: {recset['@id']}")
    if 'field' in recset:
        fields = recset['field']
        # fields could be List or Dict
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field['@id']}")
            else:
                print(f"    - Field @id: {field}")


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: automatically discover and load all record sets into pandas DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    # The record_set_id must be used by @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"No records found for record set @id: {record_set_id}")

# Choose the main record set for analysis (choose the first for demonstration)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main record set selected for analysis: {main_record_set_id}\n")
    print(f"Fields: {dataframes[main_record_set_id].columns.tolist()}")
else:
    main_record_set_id = None


## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section includes:
- Removing or filtering outliers
- Transforming numeric data
- Grouping or aggregating records by key attributes, always referencing field `@id`s

In [ ]:
# Identify a numeric field to analyze (choose a column with numeric data)
# This needs to match the actual field @id loaded in the DataFrame

import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Find numeric columns
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Pick first for demonstration
        print(f"Using numeric field for analysis: {numeric_field}")
        
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} above mean ({threshold:.2f}): {len(filtered_df)} records\n")
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Group by a category field, if available (choose first object column that is not the numeric field)
        grouping_candidates = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field]
        if grouping_candidates:
            group_field = grouping_candidates[0]
            print(f"\nGrouping filtered data by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            group_field = None
            print("No suitable field for grouping data by category.")
    else:
        print("No numeric fields found to perform EDA.")
        numeric_field, group_field = None, None
else:
    print("No record sets available to perform EDA.")
    numeric_field, group_field = None, None

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Uses matplotlib for quick plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution, if available
if main_record_set_id and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=dataframes[main_record_set_id], x=numeric_field, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field:
        # Boxplot by category
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=dataframes[main_record_set_id], x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook provided a programmatic walkthrough of the mass spectrometry protein abundance dataset using `mlcroissant`, referencing all entities by their `@id` fields. We:
- Loaded and explored record sets and fields (@id references)
- Loaded data into pandas DataFrames for each record set
- Performed basic EDA: numeric filtering, normalization, and group-wise summaries
- Visualized distributions using matplotlib/seaborn

**Key next steps:** customize the field and groupings by referencing the correct record set and field `@id`s, and apply further domain-specific analyses relevant to mass spectrometry proteomics and extracellular vesicles.